In [ ]:
from imblearn.pipeline import Pipeline
import pickle
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, auc
from sklearn.model_selection import RandomizedSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import FunctionTransformer
from scipy.stats import randint, uniform
from sklearn.experimental import enable_halving_search_cv  # aktifkan dulu
from sklearn.model_selection import HalvingRandomSearchCV

In [ ]:
df = pd.read_csv('../data/Customer-Churn-Records.csv')

In [ ]:
#Merubah ke numpy.ndarray
y = df['Exited']
X = df.drop(['Exited', 'CustomerId', 'Surname', 'Complain'], axis = 1)

In [ ]:
# 1. Tentukan kolom kategorikal dan kolom yang akan di-drop
categorical_cols = ['Geography', 'Gender', 'CardType']

# 2. Preprocessor: encoder untuk kategorikal + drop kolom lain
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ],
    remainder='passthrough'  # kolom lain (numerik) dilewatkan begitu saja
)

In [ ]:
X_train,X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)

In [ ]:
param = {
    'model__criterion': ['friedman_mse', 'squared_error'],
    'model__learning_rate': uniform(0.01, 0.5),
    'model__n_estimators': randint(50, 1000),
    'model__max_depth': randint(3, 30),
    'model__min_samples_split': randint(2, 40),
    'model__min_samples_leaf': randint(1, 12),
    'model__max_features': [None, 'sqrt', 'log2'],
    'model__subsample': uniform (0.5, 0.5)
}

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', GradientBoostingClassifier(random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rand_search = RandomizedSearchCV(pipeline, param_distributions=param, n_iter=50, cv=cv, scoring='accuracy', n_jobs=-1, verbose=2, error_score='raise')
rand_search.fit(X_train, y_train)

In [ ]:
best_pipeline = rand_search.best_estimator_
predict = best_pipeline.predict(X_test)

cmatrix = confusion_matrix(y_test, predict)
acc_score = accuracy_score(y_test, predict)

In [ ]:
sns.heatmap(cmatrix, annot=True, fmt='d', cmap='Blues', linecolor='#eaeaea',)
fig = plt.gcf()
fig.patch.set_facecolor('none')  # transparan

# Judul
plt.title(f'Confusion Matrix\nAccuracy: {acc_score:.2f}', color='#eaeaea')

# Label axis
plt.xlabel('Predicted', color='#eaeaea')
plt.ylabel('Actual', color='#eaeaea')

# Ticks
plt.xticks(color='#eaeaea')
plt.yticks(color='#eaeaea')

# Colorbar
colorbar = plt.gca().collections[0].colorbar
colorbar.set_label('Jumlah', color='#eaeaea')
colorbar.ax.tick_params(labelcolor='#eaeaea')

plt.tight_layout()
plt.show()

In [ ]:
classification_gbm = classification_report(y_test, predict)

print(classification_gbm)


In [ ]:
# ROC and AUC for Gradient Boosting
y_pred_proba_gb = best_pipeline.predict_proba(X_test)[:, 1]
fpr_gb, tpr_gb, _ = roc_curve(y_test, y_pred_proba_gb)
roc_auc_gb = auc(fpr_gb, tpr_gb)
roc_auc_score_gb = roc_auc_score(y_test, y_pred_proba_gb)

# Plotting ROC Curve Gradient Boosting Machine
plt.figure(figsize=(8, 6))
plt.plot(fpr_gb, tpr_gb, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc_gb:.2f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random Guess')

fig = plt.gcf()

# Set background color for the figure and axes
fig.patch.set_facecolor('none')  # Transparent background for saving
plt.gca().set_facecolor('#2E2E2E')  # Dark grey or black background for the plot area

# Customize axis labels and title colors for dark mode
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', color='#eaeaea')
plt.ylabel('True Positive Rate', color='#eaeaea')
plt.title('Receiver Operating Characteristic (ROC) - Gradient Boosting Machine', color='#eaeaea')

# Customize ticks color for dark mode
plt.xticks(color='#eaeaea')
plt.yticks(color='#eaeaea')

# Add legend and grid
plt.legend(loc='lower right')
plt.grid(True, color='#444444')  # Light gray grid lines for better visibility

plt.tight_layout()
plt.show()

In [ ]:
# Simpan model ke file
with open('../churnapp/churn_model_optimasi.pkl', 'wb') as modelFile:
    pickle.dump(best_pipeline, modelFile)